In [1]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from ipywidgets import interact, interact_manual, widgets, Layout, VBox, HBox, Output
from IPython.display import display, clear_output


def load_data():
    data_path = "cleaned_esg_data.csv"  
    df = pd.read_csv(data_path)
    if 'CarbonIntensity' not in df.columns and 'CarbonEmissions' in df.columns and 'Revenue' in df.columns:
        df['CarbonIntensity'] = df['CarbonEmissions'] / df['Revenue']
    df['WaterIntensity'] = df['WaterUsage'] / df['Revenue']
    df['EnergyIntensity'] = df['EnergyConsumption'] / df['Revenue']
    return df

df = load_data()


industries = sorted(df['Industry'].dropna().unique())
regions = sorted(df['Region'].dropna().unique())
min_year, max_year = int(df['Year'].min()), int(df['Year'].max())

industry_widget = widgets.SelectMultiple(
    options=industries,
    value=industries[:3],
    description='Industry',
    layout=Layout(width='50%')
)
year_slider = widgets.IntRangeSlider(
    value=[min_year, max_year],
    min=min_year,
    max=max_year,
    step=1,
    description='Year Range',
    continuous_update=False
)
region_widget = widgets.SelectMultiple(
    options=regions,
    value=regions,
    description='Region',
    layout=Layout(width='50%')
)

x_axis_widget = widgets.Dropdown(
    options=['CarbonIntensity', 'WaterIntensity', 'EnergyIntensity', 'Revenue',
             'ESG_Environmental', 'ESG_Social', 'ESG_Governance'],
    value='CarbonIntensity',
    description='X-axis'
)
y_axis_widget = widgets.Dropdown(
    options=['ProfitMargin', 'ESG_Overall', 'MarketCap', 'GrowthRate'],
    value='ProfitMargin',
    description='Y-axis'
)
color_by_widget = widgets.Dropdown(
    options=['Industry', 'Region', 'Year'],
    value='Industry',
    description='Color by'
)


output = widgets.Output()


def filter_data(industries, year_range, regions):
    mask = (df['Industry'].isin(industries)) & \
           (df['Year'].between(year_range[0], year_range[1])) & \
           (df['Region'].isin(regions))
    return df[mask].copy()


def update_plots(industries, year_range, regions, x_axis, y_axis, color_by):
    with output:
        clear_output(wait=True)
        filtered = filter_data(industries, year_range, regions)
        if filtered.empty:
            print("No data matches the current filters. Please adjust.")
            return
        
        
        avg_margin = filtered['ProfitMargin'].mean()
        avg_intensity = filtered['CarbonIntensity'].mean()
        avg_esg = filtered['ESG_Environmental'].mean()
        avg_water = filtered['WaterIntensity'].mean()
        print(f" Avg Profit Margin: {avg_margin:.1f}%")
        print(f" Avg Carbon Intensity: {avg_intensity:.2f} tons/$M")
        print(f" Avg ESG Environmental: {avg_esg:.1f}")
        print(f" Avg Water Intensity: {avg_water:.2f} m³/$M")
        print("\n" + "="*50 + "\n")
        
        
        fig_scatter = px.scatter(
            filtered, x=x_axis, y=y_axis, color=color_by,
            size='Revenue', hover_data=['CompanyName', 'Year'],
            title=f"{y_axis} vs {x_axis}"
        )
        fig_scatter.show()
        
        
        trend = filtered.groupby(['Year', 'Industry'])[['ProfitMargin', 'CarbonIntensity', 'ESG_Environmental']].mean().reset_index()
        fig_margin = px.line(trend, x='Year', y='ProfitMargin', color='Industry', title='Profit Margin Trend')
        fig_carbon = px.line(trend, x='Year', y='CarbonIntensity', color='Industry', title='Carbon Intensity Trend')
        fig_esg = px.line(trend, x='Year', y='ESG_Environmental', color='Industry', title='ESG Environmental Trend')
        fig_margin.show()
        fig_carbon.show()
        fig_esg.show()
        
        
        numeric_cols = ['ProfitMargin', 'CarbonIntensity', 'WaterIntensity', 'EnergyIntensity',
                        'ESG_Overall', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance',
                        'Revenue', 'MarketCap', 'GrowthRate']
        corr = filtered[numeric_cols].corr()
        fig_corr = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale='RdBu_r', title='Correlation Matrix')
        fig_corr.show()
        
        
        fig_box_margin = px.box(filtered, x='Industry', y='ProfitMargin', color='Industry', title='Profit Margin Distribution by Industry')
        fig_box_carbon = px.box(filtered, x='Industry', y='CarbonIntensity', color='Industry', title='Carbon Intensity Distribution by Industry')
        fig_box_margin.show()
        fig_box_carbon.show()
        
        
        region_summary = filtered.groupby('Region').agg(
            avg_profit_margin=('ProfitMargin', 'mean'),
            avg_carbon_intensity=('CarbonIntensity', 'mean')
        ).reset_index()
        fig_region = px.bar(region_summary, x='Region', y='avg_profit_margin',
                            color='avg_carbon_intensity',
                            title='Average Profit Margin by Region (colored by Carbon Intensity)')
        fig_region.show()
        
        
        summary = filtered.groupby(['Industry', 'Region']).agg(
            avg_profit_margin=('ProfitMargin', 'mean'),
            avg_carbon_intensity=('CarbonIntensity', 'mean'),
            avg_esg_env=('ESG_Environmental', 'mean'),
            count=('CompanyID', 'count')
        ).reset_index()
        display(summary)
        
        
        display(filtered)


controls = VBox([
    industry_widget,
    year_slider,
    region_widget,
    x_axis_widget,
    y_axis_widget,
    color_by_widget
])

ui = HBox([controls, output])


def on_change(change):
    update_plots(industry_widget.value, year_slider.value, region_widget.value,
                 x_axis_widget.value, y_axis_widget.value, color_by_widget.value)


for w in [industry_widget, year_slider, region_widget, x_axis_widget, y_axis_widget, color_by_widget]:
    w.observe(on_change, names='value')


display(ui)
update_plots(industry_widget.value, year_slider.value, region_widget.value,
             x_axis_widget.value, y_axis_widget.value, color_by_widget.value)